<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/fabiobento/lab-cont-2026-2/blob/main/modulo1_introducao/labs/lab02_laplace_e_ft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

# Lab 02 — Transformada de Laplace e Função de Transferência

**Módulo 01 · Semana 3 · Laboratório de Controle Automático (Ifes - Campus Guarapari)**

Neste laboratório você vai:
1. Verificar as transformadas da tabela por simulação;
2. Reproduzir os três exemplos de frações parciais da teoria;
3. Confirmar o truque dos resíduos e o caso de polos repetidos.

**Teoria de apoio:** `teoria_modulo1.md`, §1.2 (tabela de TL, frações parciais, resíduos).

In [ ]:
# %% IMPORTS (rode esta célula primeiro)
!pip install --quiet control==0.10.2
import control as ct
import numpy as np
import matplotlib.pyplot as plt
s = ct.tf('s')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
print('python-control', ct.__version__)

## Parte 1 — A tabela de TL em ação

Os pares mais usados do curso: degrau ↔ 1/s, e^(at) ↔ 1/(s−a),
sen(ωt) ↔ ω/(s²+ω²), e^(at)sen(ωt) ↔ ω/((s−a)²+ω²).
Vamos visualizar a resposta ao **impulso** (que "revela" a própria FT, pois U(s) = 1)
de alguns blocos da tabela e comparar com as expressões no tempo.

In [ ]:
t = np.linspace(0, 10, 1000)
blocos = {
    'e^{-t}': ct.tf(1, [1, 1]),
    'e^{-2t}·sen(3t)': 3*ct.tf(1, [1, 4, 13]),   # 3/((s+2)^2+9)
    'sen(3t)': 3*ct.tf(1, [1, 0, 9]),
}
for nome, G in blocos.items():
    tt, yy = ct.impulse_response(G, t)
    plt.plot(tt, yy, label=nome)
plt.xlabel('t (s)'); plt.ylabel('f(t)'); plt.legend()
plt.title('Resposta ao impulso = própria f(t) da tabela'); plt.show()

## Parte 2 — Exemplo guiado: as três expansões da teoria (§1.2.3)

**Exemplo 1 :** Y(s) = 2/[s(s+1)(s+2)] → y(t) = 1 − 2e^(−t) + e^(−2t)

**Exemplo 2:** Y(s) = 20/[s(s+1)(s+10)] → y(t) = 2 − (20/9)e^(−t) + (2/9)e^(−10t)

**Exemplo 3 (polos repetidos):** Y(s) = 2/[(s+1)²(s+2)] → y(t) = −2e^(−t) + 2t·e^(−t) + 2e^(−2t)

O código abaixo simula Y(s) como resposta ao impulso e **compara ponto a ponto**
com a expressão analítica. Se as curvas se sobrepõem, sua expansão está certa —
use esse mesmo roteiro para conferir TODA fração parcial que você fizer no papel!

In [ ]:
t = np.linspace(0, 8, 800)

# Exemplo 1
YA = 2/(s*(s+1)*(s+2))
tt, ysim = ct.impulse_response(YA, t)
yan = 1 - 2*np.exp(-tt) + np.exp(-2*tt)
plt.plot(tt, ysim, lw=3, label='simulação Y(s)')
plt.plot(tt, yan, '--', label='1 − 2e^{−t} + e^{−2t}')
plt.legend(); plt.title('Exemplo A'); plt.show()

# Exemplo 2
YB = 20/(s*(s+1)*(s+10))
tt, ysim = ct.impulse_response(YB, t)
yan = 2 - (20/9)*np.exp(-tt) + (2/9)*np.exp(-10*tt)
plt.plot(tt, ysim, lw=3, label='simulação Y(s)')
plt.plot(tt, yan, '--', label='2 − (20/9)e^{−t} + (2/9)e^{−10t}')
plt.legend(); plt.title('Exemplo B'); plt.show()

# Exemplo 3 (polos repetidos)
YC = 2/((s+1)**2*(s+2))
tt, ysim = ct.impulse_response(YC, t)
yan = -2*np.exp(-tt) + 2*tt*np.exp(-tt) + 2*np.exp(-2*tt)
plt.plot(tt, ysim, lw=3, label='simulação Y(s)')
plt.plot(tt, yan, '--', label='−2e^{−t} + 2t·e^{−t} + 2e^{−2t}')
plt.legend(); plt.title('Exemplo C — polos repetidos'); plt.show()

## Parte 3 — Sua vez

**Exercício L2.1.** No papel, expanda Y(s) = 10/[s(s+2)(s+5)] em frações parciais
(use o truque dos resíduos). Depois confira aqui, montando Y(s) e comparando com
sua expressão analítica — como na Parte 2. (Gabarito na teoria, Ex. resolvido L2.1 →
ver `exercicios_resolvidos_modulo1.md`, Exercício 1.2.1 — método idêntico.)

**Exercício L2.2.** Aplique o degrau em G(s) = 20/[(s+1)(s+10)] (isto é, simule
`ct.step_response` de G, não o impulso de Y). Qual o valor final? Confira pelo TVF
e pelo ganho DC `ct.dcgain(G)`. (§1.2.4 e §1.3.5)

**Exercício L2.3.** Um colega expandiu Y(s) = 4/[s(s+1)²] e obteve
y(t) = 4 − 4e^(−t) − 4t·e^(−t). Ele está certo? Verifique numericamente aqui e,
se estiver errado, refaça a expansão no papel mostrando onde está o erro.